<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom: 50px;">
<h1>Time Series Analysis and Forecasting</h1>
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Part D: Deep Learning Approaches</h2>
<h2>Notebook D05: Specialised Deep Learning Architectures</h2>
</div>

Every model in Part D so far was assembled by hand from PyTorch primitives. That is the right way to
understand an architecture and the wrong way to get work done: the training loop, the early stopping, the
scaling and the windowing were rewritten in each notebook, and none of it was the interesting part.

This notebook uses a library of published, forecasting-specific architectures instead, and asks the
question the whole of Part D has been building towards: does the state of the art beat the two-layer CNN
that trains in twenty seconds?

> This notebook needs NeuralForecast: `uv sync --group dl`.

---

**Contents**

1. [Imports and a Library Instead of a Loop](#1.-Imports-and-a-Library-Instead-of-a-Loop)
2. [The Landscape](#2.-The-Landscape)
3. [N-BEATS: Basis Expansion](#3.-N-BEATS:-Basis-Expansion)
4. [N-HiTS: Sampling at Several Rates](#4.-N-HiTS:-Sampling-at-Several-Rates)
5. [The Comparison](#5.-The-Comparison)
6. [The Ones We Did Not Run](#6.-The-Ones-We-Did-Not-Run)
7. [Choosing Among Them](#7.-Choosing-Among-Them)

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="1.-Imports-and-a-Library-Instead-of-a-Loop">1. Imports and a Library Instead of a Loop</h3>
</div>

In [ ]:
import importlib.util
import logging
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error

import nb_config

sns.set_theme(style="whitegrid")

NEURALFORECAST_AVAILABLE = importlib.util.find_spec("neuralforecast") is not None

if NEURALFORECAST_AVAILABLE:
    import torch
    from neuralforecast import NeuralForecast
    from neuralforecast.models import NBEATS, NHITS

    torch.set_num_threads(1)
    # The training backend is chatty; we only want the results
    logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
    logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)
    warnings.filterwarnings("ignore")

    print("NeuralForecast is available.")
else:
    print("NeuralForecast is not installed. Run 'uv sync --group dl' to follow this notebook.")

The same series, windows and horizon as the rest of Part D, so the results are directly comparable.

One thing changes: the **data format**. NeuralForecast, like most forecasting libraries, expects a long
table with three columns — `unique_id` identifying the series, `ds` for the timestamp, and `y` for the
value. That convention exists because these libraries are built to fit thousands of series at once, which
is the setting where they earn their keep. Ours is a single series, so `unique_id` is a constant.

In [ ]:
ops = pd.read_parquet(nb_config.OPS_15M_PATH)

load = (
    ops[(ops["country"] == "AT") & (ops["measure"] == "actual_entsoe_transparency")]["value"]
    .tz_convert(None)
    .resample("h").mean()
    .dropna()
    .asfreq("h")
    .loc["2016-01-01":"2019-12-31"]
)

LOOKBACK, HORIZON = 168, 24

# ── Run size ──────────────────────────────────────────────────────────────────
# As in D02 and D04: the default finishes in a few minutes, FULL_RUN trains for
# longer and reproduces the numbers quoted in the text.
FULL_RUN = False

MAX_STEPS = 1000 if FULL_RUN else 300

n_observations = len(load)
TEST_HOURS = VALIDATION_HOURS = 24 * 90
train_end = n_observations - TEST_HOURS - VALIDATION_HOURS

# The long format every forecasting library expects
frame = pd.DataFrame({
    "unique_id": "AT",
    "ds": load.index,
    "y": load.values.astype(np.float32),
})

fitting_frame = frame.iloc[:train_end + VALIDATION_HOURS]

print(f"{'FULL' if FULL_RUN else 'WORKSHOP'} run: {MAX_STEPS} training steps per model")
print(f"{len(frame):,} rows in long format")
frame.head(3)

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="2.-The-Landscape">2. The Landscape</h3>
</div>

Six architectures are worth knowing by name. They divide into three families by what problem they were
designed to solve.

**Built on basis expansion, not sequence modelling**

- **N-BEATS** stacks blocks that each fit a basis function to the input and subtract what they explained,
  passing the remainder on. There is no recurrence and no attention. It won the M4 competition and remains
  a benchmark that newer models are expected to beat.
- **N-HiTS** adds multi-rate sampling to the same idea, so different blocks work at different
  resolutions. Designed for long horizons, where N-BEATS becomes expensive.

**Transformer variants that avoid full attention**

- **Informer** replaces dense attention with a sparse approximation, so cost grows near-linearly rather
  than quadratically with the window. The direct answer to the problem Notebook
  [D04](./D04_Transformers.ipynb) ran into.
- **Autoformer** decomposes the series into trend and season inside the model, and replaces dot-product
  attention with an auto-correlation mechanism that works on periodicity directly.

**Built for the shape of real forecasting problems**

- **DeepAR** is an autoregressive RNN that outputs a *distribution* per step rather than a point, trained
  across many related series at once. It is the industrial workhorse for demand forecasting, and it is
  Notebook [B04](./B04_Probabilistic_forecasting.ipynb)'s subject in neural form.
- **Temporal Fusion Transformer** combines LSTM encoding, attention and gating, and is explicitly built
  to handle static metadata, known-future inputs and observed inputs as three distinct kinds of
  information, with variable selection you can read.

We run the first two. The rest are discussed in section 6, with reasons.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="3.-N-BEATS:-Basis-Expansion">3. N-BEATS: Basis Expansion</h3>
</div>

**N-BEATS** is the most interesting architecture in Part D, because it throws away the assumption every
other model here is built on. There is no recurrence, no convolution and no attention. It is a deep stack
of fully connected blocks, and its power comes from *what each block does with its output*.

Each block produces two things: a **backcast**, its attempt to reconstruct the input window, and a
**forecast**, its contribution to the prediction. The backcast is subtracted from the input before the
next block sees it, so each block works on the residual its predecessors could not explain, and the final
forecast is the sum of every block's contribution.

That is gradient boosting's idea, from Notebook [C02](./C02_Machine_learning_models.ipynb), rebuilt inside
a neural network: fit, subtract what you explained, pass on the remainder.

The interpretable variant constrains blocks to particular basis functions, polynomials for trend and
sinusoids for seasonality, which makes the decomposition readable in the same way Notebook A06's STL
decomposition was.

> **Training takes a few minutes.**

In [ ]:
def rolling_day_ahead(nf, model_name, frame, start, stop, horizon=HORIZON):
    """Forecast one day at a time, refreshing history without refitting.

    This is the rolling-origin evaluation of Notebook A06, applied to a neural
    model: each forecast sees every observation up to its own origin, and none
    after it.
    """
    actuals, predictions = [], []

    for origin in range(start, stop, horizon):
        history = frame.iloc[:origin]
        forecast = nf.predict(df=history)
        predictions.append(forecast[model_name].values[:horizon])
        actuals.append(frame["y"].values[origin:origin + horizon])

    return np.concatenate(actuals), np.concatenate(predictions)


def fit_and_score(model_class, model_name, max_steps=None):
    """Fit one NeuralForecast model and score it over the test period."""
    max_steps = MAX_STEPS if max_steps is None else max_steps
    started = time.time()

    nf = NeuralForecast(
        models=[model_class(h=HORIZON, input_size=LOOKBACK, max_steps=max_steps,
                            enable_progress_bar=False, random_seed=0)],
        freq="h",
    )
    nf.fit(fitting_frame, val_size=VALIDATION_HOURS)

    actuals, predictions = rolling_day_ahead(
        nf, model_name, frame,
        start=train_end + VALIDATION_HOURS,
        stop=n_observations - HORIZON,
    )

    return {
        "model": nf,
        "mae": mean_absolute_error(actuals, predictions),
        "seconds": time.time() - started,
        "actuals": actuals,
        "predictions": predictions,
    }


if NEURALFORECAST_AVAILABLE:
    nbeats = fit_and_score(NBEATS, "NBEATS")
    print(f"N-BEATS: test MAE {nbeats['mae']:.1f} MW  ({nbeats['seconds']:.0f}s)")

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="4.-N-HiTS:-Sampling-at-Several-Rates">4. N-HiTS: Sampling at Several Rates</h3>
</div>

**N-HiTS** keeps N-BEATS's residual stack and changes how each block sees the input. Blocks pool the
window at different rates: one looks at a heavily downsampled version and captures slow movement, another
looks at close to full resolution and captures fast detail. Each block's forecast is then interpolated
back up to the full horizon.

The effect is a division of labour by frequency, which is the same idea as the dilation schedule in
Notebook [D03](./D03_Convolutional_networks.ipynb) reached by a different route. It makes long horizons
much cheaper, because the low-frequency blocks work on far fewer points.

In [ ]:
if NEURALFORECAST_AVAILABLE:
    nhits = fit_and_score(NHITS, "NHITS")
    print(f"N-HiTS:  test MAE {nhits['mae']:.1f} MW  ({nhits['seconds']:.0f}s)")

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="5.-The-Comparison">5. The Comparison</h3>
</div>

Every sequence model in Part D, with the statistical baselines that started it.

Two caveats on comparability, and both matter.

**The protocol differs.** The hand-built models in D02 to D04 produced a single fixed-origin forecast for
each of 2,136 overlapping windows. These two roll the origin forward a day at a time, refreshing history
without refitting, which is closer to how a model is actually used and slightly different arithmetic. The
naive baseline is recomputed the same way, so the comparison *within* this notebook is exact; across
notebooks, the 2 MW gap between the two naive figures (561.3 here, 563.3 there) is a fair measure of how
much the protocol is worth.

**The training budgets differ.** The figures carried over from D02, D03 and D04 are full runs of those
notebooks. The two models here are trained at the workshop setting, 300 steps. That makes the comparison
conservative in this notebook's favour rather than flattering: N-BEATS is doing this on the smaller budget.
Set `FULL_RUN = True` to see how much the extra training is worth.

In [ ]:
if NEURALFORECAST_AVAILABLE:
    # The baseline, computed over exactly the same evaluation points
    naive_actuals, naive_predictions = [], []
    for origin in range(train_end + VALIDATION_HOURS, n_observations - HORIZON, HORIZON):
        naive_actuals.append(frame["y"].values[origin:origin + HORIZON])
        naive_predictions.append(frame["y"].values[origin - 24:origin - 24 + HORIZON])

    naive_mae = mean_absolute_error(
        np.concatenate(naive_actuals), np.concatenate(naive_predictions)
    )

    comparison = pd.DataFrame([
        {"Model": "N-HiTS", "Test MAE": nhits["mae"], "Family": "Specialised", "Notebook": "D05"},
        {"Model": "N-BEATS", "Test MAE": nbeats["mae"], "Family": "Specialised", "Notebook": "D05"},
        {"Model": "TCN", "Test MAE": 260.6, "Family": "Convolutional", "Notebook": "D03"},
        {"Model": "Simple CNN", "Test MAE": 291.6, "Family": "Convolutional", "Notebook": "D03"},
        {"Model": "Transformer", "Test MAE": 304.2, "Family": "Attention", "Notebook": "D04"},
        {"Model": "LSTM stacked", "Test MAE": 350.4, "Family": "Recurrent", "Notebook": "D02"},
        {"Model": "LSTM", "Test MAE": 368.6, "Family": "Recurrent", "Notebook": "D02"},
        {"Model": "Naive", "Test MAE": naive_mae, "Family": "Baseline", "Notebook": "A05"},
    ]).sort_values("Test MAE").reset_index(drop=True)

comparison.round(1)

In [ ]:
if NEURALFORECAST_AVAILABLE:
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.5),
                             gridspec_kw={"width_ratios": [1.1, 1]})

    palette = {"Specialised": "seagreen", "Convolutional": "steelblue",
               "Attention": "mediumpurple", "Recurrent": "darkorange", "Baseline": "crimson"}
    ordered = comparison.iloc[::-1]
    axes[0].barh(ordered["Model"], ordered["Test MAE"],
                 color=[palette[f] for f in ordered["Family"]])
    for position, value in enumerate(ordered["Test MAE"]):
        axes[0].text(value + 5, position, f"{value:.0f}", va="center", fontsize=9)
    axes[0].set_title("Day-ahead MAE across Part D", fontsize=13, fontweight="bold")
    axes[0].set_xlabel("MW")
    axes[0].set_xlim(0, ordered["Test MAE"].max() * 1.12)

    day = slice(0, HORIZON * 3)
    axes[1].plot(nbeats["actuals"][day], color="black", linewidth=1.8, label="Actual")
    axes[1].plot(nbeats["predictions"][day], color="seagreen", linewidth=1.3,
                 linestyle="--", label="N-BEATS")
    axes[1].plot(nhits["predictions"][day], color="mediumseagreen", linewidth=1.3,
                 linestyle=":", label="N-HiTS")
    axes[1].set_title("Three consecutive days", fontsize=13, fontweight="bold")
    axes[1].set_xlabel("Hours")
    axes[1].set_ylabel("Load (MW)")
    axes[1].legend(fontsize=9)

    for ax in axes:
        ax.grid(linestyle="--", alpha=0.4)

    plt.tight_layout()
    plt.show()

**Yes, the specialised architecture wins.** N-BEATS reaches 225.6 MW against the TCN's 260.6, a 13%
improvement on the best model Part D had produced, and N-HiTS is close behind at 236.1.

More striking than the accuracy is the cost. N-BEATS trained in about 75 seconds, where the TCN needed
nearly five minutes and the transformer around nine. It is also by far the largest model here, 2.8 million
parameters against the TCN's 41,000. Size and training time are not the same thing: N-BEATS is a stack of
dense layers with no sequential dependency to serialise and no quadratic attention matrix to build, so its
parameters are used in exactly the operation hardware is best at.

Two caveats before taking the number at face value.

**The evaluation protocol differs.** D02 to D04 scored a fixed forecast for each of 2,136 overlapping
windows; these two roll the origin forward a day at a time, refreshing history without refitting. That is
closer to real use and slightly different arithmetic, which is why the naive baseline recomputes to 561.3
here against 563.3 there. The 2 MW gap is a fair measure of how much the protocol matters, and it is much
smaller than the differences we are discussing.

**The library did some of the work.** NeuralForecast supplied the scaling, the validation split, the early
stopping and a set of defaults chosen by people who know these models well. Part of the gap between this
notebook and the hand-built ones is architecture, and part is engineering that we wrote ourselves, less
well, four notebooks in a row. That is an argument for using a library, not against the comparison.

**Exercise.** Both models here were trained at the workshop budget of 300 steps. Refit N-BEATS at 1000 steps and compare. Then do it again for three different `random_seed` values at each budget, and report the mean and the spread rather than single runs. Does more training help, and is one run enough to tell?

In [ ]:
# Your solution here


**Exercise.** The table in section 7 says to reach for N-HiTS when the horizon runs to hundreds of steps. Test it: refit both models with `h=168`, a week ahead instead of a day, and score them against a baseline that repeats the previous week. Does the ordering change, and is either model worth having at that horizon?

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="6.-The-Ones-We-Did-Not-Run">6. The Ones We Did Not Run</h3>
</div>

Four architectures from section 2 were not run, and the reasons differ.

**Informer and Autoformer** are available in NeuralForecast and would run here, but both are designed for
problems this one is not. Their contribution is making attention affordable on windows of thousands of
steps; ours is 168, where D04 showed dense attention is perfectly tractable and still loses to a small
CNN. Reaching for them on this series would demonstrate the API rather than the architecture.

**DeepAR** would be genuinely interesting, because it forecasts a distribution rather than a point, and
Notebook [B04](./B04_Probabilistic_forecasting.ipynb) established why that matters. It is a natural
extension for anyone continuing past this course: train it, then apply B04's coverage and CRPS checks to
what it produces.

**Temporal Fusion Transformer** needs what this dataset does not have. Its design is about combining
static metadata, known-future covariates and observed inputs across many related series. With one series
and no covariates, most of the model is inactive, and its selling point — readable variable selection —
has nothing to select from. The Rossmann data from Part C, with its promotions and holidays across 1,115
stores, is where it would earn its place.

That last point generalises. **These architectures are answers to specific problems**, and running one on
a problem it was not designed for tells you very little. The question to ask first is not which model is
best, but which of your constraints the model was built to address.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="7.-Choosing-Among-Them">7. Choosing Among Them</h3>
</div>

| Model | Built for | Reach for it when |
|---|---|---|
| **N-BEATS** | Accuracy on a single series | Almost always; the first thing to try |
| **N-HiTS** | Long horizons | The horizon runs to hundreds of steps |
| **Informer** | Very long inputs | Windows of thousands of steps |
| **Autoformer** | Strong periodicity, long inputs | Seasonal structure dominates |
| **DeepAR** | Many related series, uncertainty | You need intervals, not points |
| **TFT** | Rich covariates, interpretability | Static metadata and known-future inputs |

What Part D as a whole established:

**More data changes everything.** The same family of methods that lost to a random forest on 734 rows in
D01 produced the best model in the course on 30,000 sequences. Nothing about deep learning is decided in
the abstract.

**Architecture matters less than fit to the problem.** The ranking here is not a ranking of sophistication.
A two-layer CNN beat a transformer; a residual stack of dense layers beat everything. Each won because its
inductive bias matched the data, not because it was newer.

**Use a library once you know what it is doing.** Four notebooks of hand-written training loops were worth
writing, and the fifth got a better result in 75 seconds. Understand the mechanism, then stop
reimplementing it.

**The baseline is still the point.** Every model in this notebook is compared against repeating yesterday,
which the best of them beats by a factor of two and a half. That number is what makes 225.6 meaningful,
and it is the same discipline Notebook A05 introduced.

---

That completes Part D, and with it the modelling parts of the course. Part E puts the whole sequence
together on one problem: preparing the data, engineering features, building baselines, fitting and
evaluating models, and deciding what is worth deploying.

**Solutions.** Worked answers to the 2 exercises above, with the reasoning behind them, are in
[D05_Specialised_architectures_solutions.ipynb](../solutions/D05_Specialised_architectures_solutions.ipynb). Try each one yourself first.
